# CEREBRO PoC — 05 Embeddings & Semantic Relationship Discovery

**Experiment:** EXP-KNOW-002  
**Stage:** Semantic Relationship Discovery  
**Input:** Persisted Knowledge Model `KM-0001`

---

## Objective

Determine whether CEREBRO can discover candidate relationships between knowledge fragments using semantic embeddings.

The previous experiment used controlled relationships to validate the Knowledge Galaxy.

This experiment introduces machine-assisted relationship discovery:

**Knowledge Fragment → Embedding → Similarity → Candidate Relationship**

### Principle

Embedding similarity does not automatically create trusted knowledge.

Machine-discovered relationships remain **candidate relationships** until validated.

This preserves the distinction between:

- observed source knowledge,
- human-defined relationships,
- machine-discovered relationships.

The original source provenance remains unchanged.

### 1 — Load KM-0001

In [1]:
from pathlib import Path
import json

repo_root = Path.cwd().parents[1]

model_path = (
    repo_root
    / "poc/data/processed/knowledge/KM-0001.json"
)

assert model_path.exists()

with open(
    model_path,
    "r",
    encoding="utf-8"
) as f:
    knowledge_model = json.load(f)

fragments = knowledge_model["fragments"]
controlled_relationships = knowledge_model["relationships"]

print("✓ KM-0001 loaded")
print("Fragments:", len(fragments))
print("Controlled relationships:", len(controlled_relationships))

✓ KM-0001 loaded
Fragments: 3
Controlled relationships: 3


## 2. Semantic Embedding

Each knowledge fragment is converted into a vector representation.

Fragments with similar semantic meaning should occupy nearby regions of the embedding space.

For this PoC, embedding is performed locally to demonstrate that semantic relationship discovery can operate without sending the user's knowledge to an external AI provider.

The embedding is a **derived representation**.

It does not replace the original knowledge fragment or its provenance.

### 2 — Configure local embedding

In [2]:
import requests

OLLAMA_URL = "http://localhost:11434"

EMBEDDING_MODEL = "mxbai-embed-large:latest"

def embed_text(text):

    response = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={
            "model": EMBEDDING_MODEL,
            "input": text
        },
        timeout=120
    )

    response.raise_for_status()

    result = response.json()

    return result["embeddings"][0]

print(
    "✓ Embedding capability configured:",
    EMBEDDING_MODEL
)

✓ Embedding capability configured: mxbai-embed-large:latest


### 3 — Smoke test

In [4]:
test_embedding = embed_text(
    "CEREBRO Digital Knowledge Twin"
)

print("✓ Embedding generated")
print("Dimensions:", len(test_embedding))
print("Sample:", test_embedding[:5])

✓ Embedding generated
Dimensions: 1024
Sample: [-0.0037215056, 0.023080077, -0.029471654, -0.0240521, -0.014847591]


## 3. Generate Knowledge Fragment Embeddings

Generate an embedding for each persisted knowledge fragment.

Each embedding remains associated with:

- Fragment ID
- Artifact ID
- Embedding model
- Original fragment provenance

This allows CEREBRO to change embedding models later without changing the identity of the underlying knowledge.

### 4 - Generate fragments embeddings

In [5]:
fragment_embeddings = {}

for fragment in fragments:

    fragment_id = fragment["fragment_id"]

    vector = embed_text(
        fragment["content"]
    )

    fragment_embeddings[fragment_id] = {
        "vector": vector,
        "model": EMBEDDING_MODEL,
        "artifact_id": fragment["artifact_id"]
    }

    print(
        f"✓ {fragment_id}: "
        f"{len(vector)} dimensions"
    )

✓ KF-0001: 1024 dimensions
✓ KF-0002: 1024 dimensions
✓ KF-0003: 1024 dimensions


## 4. Measure Semantic Similarity

CEREBRO compares fragment embeddings using cosine similarity.

Higher similarity indicates that two fragments occupy closer positions in semantic space.

Similarity alone does **not** prove that a meaningful knowledge relationship exists.

It is therefore treated as evidence for a candidate connection rather than as a trusted relationship.

### 5 — Cosine similarity

In [6]:
import numpy as np

def cosine_similarity(a, b):

    a = np.array(a)
    b = np.array(b)

    return float(
        np.dot(a, b)
        /
        (
            np.linalg.norm(a)
            * np.linalg.norm(b)
        )
    )

### 6 — Compare every pair

In [7]:
from itertools import combinations

similarity_results = []

for left, right in combinations(
    fragments,
    2
):

    left_id = left["fragment_id"]
    right_id = right["fragment_id"]

    score = cosine_similarity(
        fragment_embeddings[left_id]["vector"],
        fragment_embeddings[right_id]["vector"]
    )

    similarity_results.append({
        "source": left_id,
        "target": right_id,
        "similarity": round(score, 4)
    })

similarity_results = sorted(
    similarity_results,
    key=lambda x: x["similarity"],
    reverse=True
)

for result in similarity_results:

    print(
        result["source"],
        "↔",
        result["target"],
        "=",
        result["similarity"]
    )

KF-0001 ↔ KF-0003 = 0.7228
KF-0002 ↔ KF-0003 = 0.6948
KF-0001 ↔ KF-0002 = 0.6841


## 5. Candidate Relationship Classification

Raw similarity scores are useful for evaluation but are not ideal for the CEREBRO user experience.

CEREBRO can translate similarity evidence into human-readable relationship indicators.

For this initial experiment, provisional thresholds are used only to visualize candidate strength.

These thresholds are experimental and are not yet production decisions.

### 7 - Candidate relationship similarity

In [8]:
def relevance_label(score):

    if score >= 0.75:
        return "Highly Related"

    elif score >= 0.55:
        return "Related"

    else:
        return "Potential Connection"


candidate_relationships = []

for index, result in enumerate(
    similarity_results,
    start=1
):

    candidate_relationships.append({
        "relationship_id":
            f"CAND-{index:04d}",

        "source":
            result["source"],

        "target":
            result["target"],

        "type":
            "SEMANTIC_SIMILARITY",

        "similarity":
            result["similarity"],

        "relevance":
            relevance_label(
                result["similarity"]
            ),

        "status":
            "candidate",

        "method":
            "embedding_cosine_similarity",

        "model":
            EMBEDDING_MODEL
    })

### 8 - Inspectc

In [9]:
for relationship in candidate_relationships:

    print(
        relationship["source"],
        "↔",
        relationship["target"],
        "|",
        relationship["relevance"],
        "| similarity:",
        relationship["similarity"]
    )

KF-0001 ↔ KF-0003 | Related | similarity: 0.7228
KF-0002 ↔ KF-0003 | Related | similarity: 0.6948
KF-0001 ↔ KF-0002 | Related | similarity: 0.6841


## 6. Compare Machine Discovery with Controlled Relationships

`KM-0001` already contains controlled relationships created during the previous experiment.

These provide a small reference against which machine-discovered connections can be compared.

The objective is not to prove embedding accuracy from three fragments.

The objective is to validate the mechanism:

**Known Relationship ↔ Machine-Discovered Semantic Connection**

A larger benchmark will be required for meaningful retrieval and relationship quality evaluation.

### 9 - Compare machine discovery with control'd relationships

In [10]:
controlled_pairs = {
    frozenset([
        relationship["source"],
        relationship["target"]
    ])
    for relationship
    in controlled_relationships
}

for candidate in candidate_relationships:

    pair = frozenset([
        candidate["source"],
        candidate["target"]
    ])

    candidate["matches_controlled_relationship"] = (
        pair in controlled_pairs
    )

    print(
        candidate["source"],
        "↔",
        candidate["target"],
        "|",
        candidate["relevance"],
        "| controlled:",
        candidate[
            "matches_controlled_relationship"
        ]
    )

KF-0001 ↔ KF-0003 | Related | controlled: True
KF-0002 ↔ KF-0003 | Related | controlled: True
KF-0001 ↔ KF-0002 | Related | controlled: True


## 7. Persist Machine-Discovered Relationships

Candidate relationships are persisted separately from trusted relationships.

This distinction prevents an embedding model from silently changing the trusted Digital Knowledge Twin.

The Galaxy may display candidate connections as an explainability layer while retaining their status and supporting evidence.

### 10 - Persist machine discovered relationships

In [11]:
output = {
    "experiment_id": "EXP-KNOW-002",

    "knowledge_model": "KM-0001",

    "embedding": {
        "provider": "local_ollama",
        "model": EMBEDDING_MODEL
    },

    "candidate_relationships":
        candidate_relationships
}

output_dir = (
    repo_root
    / "poc/data/processed/knowledge"
)

output_path = (
    output_dir
    / "KM-0001-semantic-candidates.json"
)

with open(
    output_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("✓ Candidate relationships persisted")
print("Output:", output_path)

✓ Candidate relationships persisted
Output: /Users/joeldizon/development/cerebro_dev/cerebro/poc/data/processed/knowledge/KM-0001-semantic-candidates.json


### 11 - Final check

In [12]:
assert len(fragment_embeddings) == len(fragments)

assert len(candidate_relationships) > 0

assert all(
    relationship["status"] == "candidate"
    for relationship in candidate_relationships
)

print("✓ Fragment embeddings generated")
print("✓ Semantic similarity calculated")
print("✓ Candidate relationships discovered")
print("✓ Human-readable relevance assigned")
print("✓ Controlled relationships preserved")
print("✓ Candidate relationships persisted separately")

print("\nEXP-KNOW-002: PASS")

✓ Fragment embeddings generated
✓ Semantic similarity calculated
✓ Candidate relationships discovered
✓ Human-readable relevance assigned
✓ Controlled relationships preserved
✓ Candidate relationships persisted separately

EXP-KNOW-002: PASS


## Experiment Conclusion

**EXP-KNOW-002 — Embeddings & Semantic Relationship Discovery: PASS**

CEREBRO successfully generated semantic embeddings for persisted knowledge fragments and used them to identify candidate relationships.

The experiment demonstrates:

**Knowledge Fragment → Local Embedding → Semantic Similarity → Candidate Connection**

Machine-discovered connections remain separate from trusted relationships and retain:

- fragment identity,
- similarity evidence,
- relevance classification,
- embedding model provenance,
- candidate status.

The experiment therefore extends the Knowledge Galaxy without allowing an embedding model to silently redefine trusted knowledge.

### Evolution of the Galaxy

**Controlled Relationships**

↓

**Machine-Discovered Candidate Relationships**

↓

**Validation**

↓

**Trusted Knowledge Relationships**

### Next Step

Use the discovered relationships to enhance the existing **Knowledge Galaxy**, showing both controlled and machine-discovered connections while preserving Assisted Recollection and source provenance.